# NEMO wind example

In this notebook, we resample the surface stress experienced by a NEMO ocean (ORCA12) onto the ECCO LLC90 grid

In [ ]:
from eccobridge.universal import rechunk
from eccobridge.nemo import resample_nemo, interpolate_masked_nemofield
from nemo_python.utils import rotate_vector
import glob
import xnemogcm
import ecco_v4_py as ecco
import xarray as xr
import xgcm
import matplotlib
import matplotlib.pyplot as plt
import cmocean
import copy
from dask.diagnostics import ProgressBar

In [ ]:
# Location of NEMO forcing data and NEMO grid information
nemo_datadir = "/gws/ssde/j25a/bas_pog/astyles/NEMO_OceanBound/renamed/"                 # Path to the directory containing the NEMO model data
nemo_datafiles = [f"{nemo_datadir}/ORCA0083-N06_{y}_monthly_tauuo_grid_U.nc" for y in range(1987,2013)] + [f"{nemo_datadir}/ORCA0083-N06_{y}_monthly_tauvo_grid_V.nc" for y in range(1987,2013)]

nemo_domdir = "/gws/ssde/j25a/nemo/vol1/ORCA0083-N006/domain/"               # Directory containing the NEMO grid information. 
nemo_domfiles = ["mesh_hgr.nc", "mesh_zgr.nc", "mask.nc"]                    # File names for the NEMO grid information. Usually domcfg files or mesh_* files
nemo_varnames = ['tauuo', 'tauvo']                                           # NEMO variable names you would like to resample

# Location of ECCO grid file
ecco_gridfile = './ECCO-GRID.nc'

# Chunking of the DataArrays to maintain sensible memory use. 
chunks_nemo = {'t':12, 'z_c':10, 'x_c':-1, 'y_c':-1, 'x_f':-1, 'y_f':-1, 'y':-1, 'x':-1}
chunks_ecco = {'i':-1, 'j':-1, 'i_g':-1, 'j_g':-1, 'tile':-1, 'lag':12}

In [ ]:
# Load the NEMO grid information using xnemogcm
ds_nemo_grid = rechunk(xnemogcm.open_domain_cfg(datadir=nemo_domdir, files=nemo_domfiles), chunks_nemo)

# Load the NEMO grid information into XGCM for grid-awareness and easy interpolation
ds_nemo_xgcmgrid = xgcm.Grid(ds_nemo_grid, metrics=xnemogcm.get_metrics(ds_nemo_grid), periodic=True)

# Also load the NEMO grid information in a less grid-aware way (for compatibilty with python_nemo routines)
ds_nemo_gridraw = rechunk(xnemogcm.domcfg.open_file_multi( [ f"{nemo_domdir}/{x}" for x in nemo_domfiles] ), chunks_nemo)

# Load the NEMO data
ds_nemo = rechunk(xnemogcm.open_nemo(ds_nemo_grid, files=nemo_datafiles), chunks_nemo) #Works if the time coordinate is consistent across the entire dataset and has correct metadata

# Load the ECCO grid information
ds_ecco_grid = rechunk(xr.open_dataset(ecco_gridfile), chunks_ecco)

In [ ]:
# Interpolate the X-stress and the Y-stress onto the T points
nemo_taux = ds_nemo_xgcmgrid.interp(ds_nemo['tauuo'], 'X').where(ds_nemo_grid['tmask'].max(dim='z_c'))
nemo_tauy = ds_nemo_xgcmgrid.interp(ds_nemo['tauvo'], 'Y').where(ds_nemo_grid['tmask'].max(dim='z_c'))

# Routines for temporarily renaming the x and y coordinates (compatibility with nemo_python routines)
rename_tmp = lambda x : x.rename(x_c='x', y_c='y')
rename_tmpr = lambda x : x.rename(x='x_c', y='y_c')

# Use the nemo_python rotate_vector subroutine to get the true zonal and meridional components
nemo_taue, nemo_taun = rotate_vector(rename_tmp(nemo_taux), rename_tmp(nemo_tauy), ds_nemo_gridraw, gtype='T')
nemo_taue, nemo_taun = rename_tmpr(nemo_taue), rename_tmpr(nemo_taun)

In [ ]:
taue_ecco = resample_nemo(nemo_taue, None,
                    ds_nemo_grid, ds_ecco_grid, 
                    resample_type='pr_gauss', radius_of_influence=120e3, reduce_data=False).rename('taue_nemo')

taun_ecco = resample_nemo(nemo_taun, None,
                    ds_nemo_grid, ds_ecco_grid, 
                    resample_type='pr_gauss', radius_of_influence=120e3, reduce_data=False).rename('taun_nemo')

# Comparison of fields

In [ ]:
# Function for selecting time index
# tsel = lambda x : x.isel(t=-1)

# NEMO fields >>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>.
# fig, ax = plt.subplots(1,2,dpi=200, figsize=(2*3*3.1554, 2*3.1554))
# z = tsel(nemo_taue)
# clim = float(abs(z).max().values)
# cax = ax[0].pcolormesh(z, vmin=-clim, vmax=+clim, cmap=cmocean.cm.balance)
# fig.colorbar(cax, orientation='horizontal', ax=ax[0])

# z = tsel(nemo_taun)
# clim = float(abs(z).max().values)
# cax = ax[1].pcolormesh(z, vmin=-clim, vmax=+clim, cmap=cmocean.cm.balance)
# fig.colorbar(cax, orientation='horizontal', ax=ax[1])

# ECCO fields >>>>>>>>>>>>>>>>>>>>>>>>>>
# z = tsel(taue_ecco).where(ds_ecco_grid.maskC.max(dim='k'), 0)
# clim = float(abs(z).max().values)
# fig, ax = ecco.plot_tiles(z, cmap=cmocean.cm.balance, \
#                 show_colorbar=True, fig_size=8,\
#                 cmin=-clim, cmax=clim,\
#                layout='latlon',rotate_to_latlon=True,\
#                show_tile_labels=False, \
#                Arctic_cap_tile_location=10)

# z = tsel(taun_ecco).where(ds_ecco_grid.maskC.max(dim='k'), 0)
# clim = float(abs(z).max().values)
# fig, ax = ecco.plot_tiles(z, cmap=cmocean.cm.balance, \
#                 show_colorbar=True, fig_size=8,\
#                 cmin=-clim, cmax=clim,\
#                layout='latlon',rotate_to_latlon=True,\
#                show_tile_labels=False, \
#                Arctic_cap_tile_location=10)

# Save fields

In [ ]:
with ProgressBar():
    taue_ecco.to_netcdf('/work/scratch-nopw2/afstyles/NEMO_resampled/ECCO_nemo_taue.1987-2012.nc')

with ProgressBar():
    taun_ecco.to_netcdf('/work/scratch-nopw2/afstyles/NEMO_resampled/ECCO_nemo_taun.1987-2012.nc')